# 실습 4: 구문 분석 & 종합 텍스트 분석 파이프라인
**소요시간: 30분** | 난이도: ⭐⭐⭐⭐

## 학습 목표
1. `detect_syntax`로 품사(POS) 태깅을 수행합니다.
2. 앞서 배운 API를 통합하는 **종합 분석 파이프라인**을 구현합니다.
3. 분석 결과를 JSON/CSV로 저장하고 시각화 대시보드를 만듭니다.

## API 개요
```python
# 구문 분석 (품사 태깅)
comprehend.detect_syntax(Text='...', LanguageCode='en')
```

### 주요 품사 태그
```
NOUN  VERB  ADJ  ADV  PROPN(고유명사)
DET   ADP   CONJ  PUNCT  NUM
```

### Syntax 결과 구조
```
{
  'TokenId': 1,
  'Text': 'The',
  'PartOfSpeech': {'Tag': 'DET', 'Score': 0.999},
  'BeginOffset': 0, 'EndOffset': 3
}
```


---
## 🏢 기업 시나리오 — 데이터팀의 VOC 자동 분석 시스템

당신은 **데이터 엔지니어**입니다.
지금까지 배운 기능(언어감지·감성·개체인식·핵심문구)을 **하나의 파이프라인**으로 묶어,
고객의 소리(VOC)를 자동 분석하는 시스템을 만듭니다.

이번 실습에서는 다음을 구현합니다.
1. **구문 분석(Syntax)** → 품사 태깅으로 텍스트 구조 파악
2. **종합 분석 함수** → 4개 API를 하나의 `analyze_text()`로 통합
3. **결과 적재 + 대시보드** → JSON/CSV 저장 및 시각화

> 💡 실무에서는 이 `analyze_text()` 파이프라인을 **AWS Lambda**로 배포해, 새 리뷰가 들어올 때마다 실시간으로 자동 분석합니다.


In [ ]:
# ✅ [제공 코드] 환경 초기화
import boto3, json, os
import matplotlib.pyplot as plt
import pandas as pd
from collections import Counter

# 한글 폰트 설정 (SageMaker Studio) — 최초 1회만 설치, 이후 즉시 로드
try:
    import koreanize_matplotlib
except ImportError:
    import sys, subprocess
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'koreanize-matplotlib'])
    import koreanize_matplotlib

comprehend = boto3.client('comprehend', region_name='us-east-1')
os.makedirs('./lab04_output', exist_ok=True)
print('✅ 환경 초기화 완료 (us-east-1)')


## ✏️ TODO 1: detect_syntax — 품사 태깅

텍스트를 형태소 단위로 분석하고 품사 분포를 시각화하세요.


In [ ]:
# ✏️ TODO 1: detect_syntax API를 호출하고 품사 분포를 출력하세요
syntax_text = (
    "Amazon Web Services provides reliable cloud computing solutions "
    "that help businesses scale quickly and efficiently."
)

response = comprehend.detect_syntax(
    Text=_____,           # ← syntax_text · 품사 분석할 원문
    LanguageCode=_____    # ← 'en' · 구문 분석(detect_syntax)은 영어 등 일부 언어만 지원
)

tokens = response[_____]  # ← 'SyntaxTokens' · 토큰(단어)별 분석 결과 리스트 (Text/PartOfSpeech 포함)
print(f'토큰 수: {len(tokens)}개')
print('-' * 50)
for tok in tokens:
    pos  = tok['PartOfSpeech'][_____]   # ← 'Tag' · 품사 태그 (NOUN명사/VERB동사/ADJ형용사/PROPN고유명사 등)
    conf = tok['PartOfSpeech'][_____]   # ← 'Score' · 품사 판정 신뢰도 (0~1)
    print(f"  {tok['Text']:<15} {pos:<8} ({conf:.3f})")


In [ ]:
# ✅ [제공 코드] 품사 분포 시각화
pos_counts = Counter(t['PartOfSpeech']['Tag'] for t in tokens)
labels = list(pos_counts.keys())
sizes  = list(pos_counts.values())

plt.figure(figsize=(9, 4))
plt.bar(labels, sizes, color='#3498db')
plt.title('품사 분포 (Part-of-Speech)')
plt.ylabel('빈도')
plt.tight_layout()
plt.show()

nouns = [t['Text'] for t in tokens if t['PartOfSpeech']['Tag'] in ('NOUN','PROPN')]
print('주요 명사/고유명사:', nouns)


## ✏️ TODO 2: 종합 분석 파이프라인 구현

언어 감지 → 감성 분석 → 개체 인식 → 핵심 문구 추출을 하나의 함수로 통합하세요.


In [ ]:
# ✏️ TODO 2: analyze_text() 통합 파이프라인 함수를 완성하세요
def analyze_text(text, lang='ko'):
    """Comprehend 종합 분석 파이프라인"""
    result = {'text': text, 'language': lang}

    # 1단계: 언어 자동 감지
    lang_resp = comprehend.detect_dominant_language(Text=_____) # ← text · 언어를 자동 감지할 입력
    detected_lang = lang_resp['Languages'][0][_____]             # ← 'LanguageCode' · 가장 유력한 언어 코드 (이후 단계에 재사용)
    result['detected_language'] = detected_lang

    # 2단계: 감성 분석
    sent_resp = comprehend.detect_sentiment(
        Text=_____, LanguageCode=_____   # ← text, detected_lang · 분석 원문과 앞에서 감지한 언어
    )
    result['sentiment']       = sent_resp[_____]          # ← 'Sentiment' · 대표 감성 (POSITIVE/NEGATIVE/NEUTRAL/MIXED)
    result['sentiment_scores']= sent_resp[_____]          # ← 'SentimentScore' · 감성별 확률 dict

    # 3단계: 개체 인식
    ent_resp = comprehend.detect_entities(
        Text=_____, LanguageCode=_____   # ← text, detected_lang · 분석 원문과 앞에서 감지한 언어
    )
    result['entities'] = [
        {'text': e['Text'], 'type': e['Type'], 'score': round(e['Score'],3)}
        for e in ent_resp[_____]         # ← 'Entities' · 감지된 개체 리스트
    ]

    # 4단계: 핵심 문구
    kp_resp = comprehend.detect_key_phrases(
        Text=_____, LanguageCode=_____   # ← text, detected_lang · 분석 원문과 앞에서 감지한 언어
    )
    result['key_phrases'] = [p['Text'] for p in kp_resp[_____]]  # ← 'KeyPhrases' · 추출된 핵심 문구 리스트

    return result

# 테스트
test = '삼성전자가 2024년 서울에서 AI 반도체 신제품을 발표했습니다.'
out  = analyze_text(test)
print(f"언어: {out['detected_language']}")
print(f"감성: {out['sentiment']}")
print(f"개체: {out['entities']}")
print(f"핵심문구: {out['key_phrases']}")


## ✏️ TODO 3: 배치 파이프라인 & 결과 저장

여러 뉴스 헤드라인을 파이프라인으로 처리하고 결과를 JSON과 CSV로 저장하세요.


In [ ]:
# ✏️ TODO 3: 여러 텍스트를 파이프라인으로 처리하고 결과를 저장하세요
headlines = [
    '현대차, 전기차 신모델 출시로 테슬라와 본격 경쟁 선언',
    '코스피 3,000선 돌파, 외국인 투자자 대규모 순매수',
    'BTS 월드투어 티켓 매진, 팬들 아쉬움 토로',
    '기상청, 이번 주말 전국 폭설 예보',
]

all_results = []
for i, h in enumerate(headlines):
    print(f'처리 중 [{i+1}/{len(headlines)}]: {h[:25]}...')
    r = _____(h)            # ← analyze_text · 앞에서 만든 통합 파이프라인 함수 호출
    all_results.append(r)

# JSON 저장
with open('./lab04_output/results.json', _____) as f:   # ← 'w' · 쓰기 모드 (write)
    json.dump(_____, f, ensure_ascii=False, indent=2)   # ← all_results · 저장할 전체 분석 결과 리스트
print('\n✅ results.json 저장 완료')

# CSV 저장
df = pd.DataFrame([{
    '헤드라인': r['text'][:30],
    '감성'    : r[_____],                                # ← 'sentiment' · analyze_text 결과의 감성 값
    '개체수'  : len(r['entities']),
    '핵심문구': ', '.join(r['key_phrases'][:3])
} for r in all_results])
df.to_csv('./lab04_output/results.csv', index=False, encoding='utf-8-sig')
print('✅ results.csv 저장 완료')
print(df.to_string(index=False))


## ✏️ TODO 4: 종합 시각화 대시보드

분석 결과를 2×2 서브플롯 대시보드로 시각화하세요.


In [ ]:
# ✏️ TODO 4: 분석 결과를 2×2 대시보드로 시각화하세요
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

# [0,0] 감성 분포 파이차트
sent_counts = Counter(r[_____] for r in all_results)  # ← 'sentiment' · 결과별 감성을 세어 분포 계산
colors_map  = {'POSITIVE':'#2ecc71','NEGATIVE':'#e74c3c','NEUTRAL':'#95a5a6','MIXED':'#f39c12'}
axes[0,0].pie(
    sent_counts.values(),
    labels=sent_counts.keys(),
    colors=[colors_map.get(k,'#95a5a6') for k in sent_counts.keys()],
    autopct='%1.0f%%'
)
axes[0,0].set_title('감성 분포')

# [0,1] 개체 유형 분포
all_ent_types = [e['type'] for r in all_results for e in r['entities']]
ent_cnt = Counter(all_ent_types)
axes[0,1].bar(ent_cnt.keys(), ent_cnt.values(), color='#3498db')
axes[0,1].set_title('개체 유형 분포')
axes[0,1].tick_params(axis='x', rotation=30)

# [1,0] 헤드라인별 개체 수
ent_counts = [len(r['entities']) for r in all_results]
short_labels = [r['text'][:12]+'...' for r in all_results]
axes[1,0].barh(short_labels, _____, color='#9b59b6')  # ← ent_counts · 헤드라인별 개체 수
axes[1,0].set_title('헤드라인별 개체 수')

# [1,1] 헤드라인별 핵심 문구 수
kp_counts = [len(r['key_phrases']) for r in all_results]
axes[1,1].bar(range(len(headlines)), _____, color='#e67e22')  # ← kp_counts · 헤드라인별 핵심 문구 수
axes[1,1].set_xticks(range(len(headlines)))
axes[1,1].set_xticklabels([f'뉴스{i+1}' for i in range(len(headlines))])
axes[1,1].set_title('헤드라인별 핵심 문구 수')

plt.suptitle('Amazon Comprehend 종합 분석 대시보드', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('./lab04_output/dashboard.png', dpi=150)
plt.show()
print('✅ dashboard.png 저장 완료')


---
## 🔗 실무로 연결하기 — 오늘 만든 것이 곧 실무 시스템

오늘 4개 실습에서 만든 기능이 모이면 아래와 같은 **실무 VOC 분석 파이프라인**이 됩니다.

`고객 리뷰·문의(S3)` → `Lambda` → `Comprehend(감성·개체·핵심문구·PII)` → `DB 적재` → `BI 대시보드`

- 마케팅팀: 감성 추이로 캠페인 반응 측정
- CS팀: 부정 급증 시 자동 알림
- 보안팀: PII 자동 마스킹으로 규정 준수

**축하합니다! 여러분은 방금 기업이 실제로 사용하는 텍스트 AI 파이프라인의 축소판을 완성했습니다.** 🎉


## 💡 심화 도전
1. 10개 이상의 뉴스 기사를 수집하여 날짜별 감성 트렌드를 꺾은선 그래프로 시각화하세요.
2. 가장 많이 언급된 ORGANIZATION(조직) Top 5를 추출하세요.
3. 파이프라인을 Lambda 함수로 배포하는 아키텍처를 설계해보세요.


## ✅ 정답 코드

👆 모두 풀고 난 후 확인하세요

```python
# TODO 1
response = comprehend.detect_syntax(Text=syntax_text, LanguageCode='en')
tokens = response['SyntaxTokens']
pos  = tok['PartOfSpeech']['Tag']
conf = tok['PartOfSpeech']['Score']

# TODO 2
lang_resp = comprehend.detect_dominant_language(Text=text)
detected_lang = lang_resp['Languages'][0]['LanguageCode']
sent_resp = comprehend.detect_sentiment(Text=text, LanguageCode=detected_lang)
result['sentiment']        = sent_resp['Sentiment']
result['sentiment_scores'] = sent_resp['SentimentScore']
ent_resp = comprehend.detect_entities(Text=text, LanguageCode=detected_lang)
result['entities'] = [... for e in ent_resp['Entities']]
kp_resp = comprehend.detect_key_phrases(Text=text, LanguageCode=detected_lang)
result['key_phrases'] = [p['Text'] for p in kp_resp['KeyPhrases']]

# TODO 3
r = analyze_text(h)
with open('./lab04_output/results.json', 'w') as f:
    json.dump(all_results, f, ensure_ascii=False, indent=2)
df = pd.DataFrame([{'감성': r['sentiment'], ...} for r in all_results])

# TODO 4
sent_counts = Counter(r['sentiment'] for r in all_results)
axes[1,0].barh(short_labels, ent_counts, color='#9b59b6')
axes[1,1].bar(range(len(headlines)), kp_counts, color='#e67e22')
```
